# Problem Statement : Spam or ham?

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib as plt 
import seaborn as sns 

from sklearn.model_selection import train_test_split 
from sklearn.feature_extraction.text import CountVectorizer 
 
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

import warnings 
warnings.filterwarnings('ignore')

# Load Data

In [2]:
spam = pd.read_csv('/Users/vydhyamvishnusai/NAVIE_BYES_IMPLEMENTATION_BY_SAI/spam.csv')
spam.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
spam.tail()

,Category,Message
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...
5571,ham,Rofl. Its true to its name


In [4]:
spam.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Category  5572 non-null   str  
 1   Message   5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [5]:
spam.isna().sum()

Category    0
Message     0
dtype: int64

# Feature Extraction: CountVectorizer

In [6]:
train_set = {"The sky is blue.", "The sun is bright."}
test_set = {"The sun in the sky is bright","we can see the shining sun, the bright sun."}


In [7]:
vectorizer = CountVectorizer()

In [8]:
vectorizer.fit(train_set)
vectorizer.vocabulary_

{'the': 5, 'sun': 4, 'is': 2, 'bright': 1, 'sky': 3, 'blue': 0}

In [9]:
vocab_dict = vectorizer.vocabulary_.copy()
dict(sorted(vocab_dict.items(), key = lambda x : x[1]))

{'blue': 0, 'bright': 1, 'is': 2, 'sky': 3, 'sun': 4, 'the': 5}

In [10]:
test_vec = vectorizer.transform(test_set)
print(test_vec)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 8 stored elements and shape (2, 6)>
  Coords	Values
  (0, 1)	1
  (0, 4)	2
  (0, 5)	2
  (1, 1)	1
  (1, 2)	1
  (1, 3)	1
  (1, 4)	1
  (1, 5)	2


• Each entry is represented as (row_index, column_index) and count

• row_index: Indicates which document (from test_set) the entry corresponds to.

• column_index: Indicates which word (from the training vocabulary) is being counted.

• count: Indicates how many times the word appears in that document.

In [11]:
test_vec.toarray()

array([[0, 1, 0, 0, 2, 2],
       [0, 1, 1, 1, 1, 2]])

In [12]:
vectorizer.inverse_transform(test_vec)

[array(['bright', 'sun', 'the'], dtype='<U6'),
 array(['bright', 'is', 'sky', 'sun', 'the'], dtype='<U6')]

We only got the token back, not the order of words. This is limitation of countvectorizer

# Data Preprocessing

In [13]:
spam['Category'] =  spam['Category'].apply(lambda x:1 if x == 'spam' else 0)
spam.head(5)

,Category,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [14]:
x = spam['Message']
y = spam['Category']

In [15]:
x.head()

0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: Message, dtype: str

In [16]:
y.head()

0    0
1    0
2    1
3    0
4    0
Name: Category, dtype: int64

In [17]:
vect = CountVectorizer(ngram_range = (1,3), min_df = 5, max_features = 8000)
x_vect = vect.fit_transform(x).toarray()
x_vect[:5]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5, 4722))

In [18]:
#split data
x_train, x_test, y_train, y_test = train_test_split(x_vect, y , test_size = 0.5, random_state = 42)

# Building Navie Byes Model

In [19]:
nb = GaussianNB()
nb.fit(x_train, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [20]:
nb.score(x_train,y_train)

0.964824120603015

In [21]:
nb.score(x_test,y_test)

0.9400574300071788

# Comparison Naview Byes with KNN and SVM

In [22]:
#build knn 
%time 
knn = KNeighborsClassifier()
knn.fit(x_train,y_train)
knn.score(x_train,y_train)
knn.score(x_test, y_test)

CPU times: user 4 μs, sys: 1 μs, total: 5 μs
Wall time: 9.06 μs


0.9149318018664753

In [23]:
%time 
svc = SVC()
svc.fit(x_train, y_train)
svc.score(x_train, y_train)
svc.score(x_test, y_test)

CPU times: user 3 μs, sys: 1 μs, total: 4 μs
Wall time: 8.82 μs


0.9824120603015075

In [24]:
%time
nb = GaussianNB()
nb.fit(x_train, y_train)
nb.score(x_train,y_train)
nb.score(x_test,y_test)

CPU times: user 4 μs, sys: 1 μs, total: 5 μs
Wall time: 7.87 μs


0.9400574300071788

# Prediction

In [25]:
sample = "You won 25 lakh"
vec = vect.transform([sample]).toarray()
nb.predict(vec)

array([1])

In [26]:
sample = "Hi sai, I need your help"
vec1 = vect.transform([sample]).toarray()
nb.predict(vec1)

array([1])

Misclassification, this shows even if we have training and testing score of 94%, we can get wrong results.

# Multinomial Navie Byes

In [29]:
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB()
nb.fit(x_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [30]:
nb.score(x_train, y_train)

0.9874371859296482

In [31]:
nb.score(x_test, y_test)

0.9856424982053122

In [32]:
email_1='Sounds great! Are you home now?'

In [35]:
vec = vect.transform([email_1]).toarray()
predictions = nb.predict(vec)
print(predictions)

[0]


In [36]:
email_2 = "Congratulations! You've been selected to receive a FREE iPhone—click here to claim your prize!"

In [37]:
vec = vect.transform([email_2]).toarray()
predictions = nb.predict(vec)
print(predictions)

[1]


In [41]:
from sklearn.metrics import confusion_matrix 
confusion_matrix(y_test, nb.predict(x_test))

array([[2401,   15],
       [  25,  345]])

In [42]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, nb.predict(x_test))

0.9856424982053122

In [43]:
from sklearn.metrics import precision_score 
precision_score(y_test, nb.predict(x_test))

0.9583333333333334

In [44]:
from sklearn.metrics import recall_score 
recall_score(y_test, nb.predict(x_test))

0.9324324324324325

In [46]:
from sklearn.metrics import f1_score 
f1_score(y_test, nb.predict(x_test))

0.9452054794520548

In [50]:
from sklearn.metrics import classification_report 
print(classification_report(y_test, nb.predict(x_test)))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      2416
           1       0.96      0.93      0.95       370

    accuracy                           0.99      2786
   macro avg       0.97      0.96      0.97      2786
weighted avg       0.99      0.99      0.99      2786

